In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver      
from selenium.webdriver.common.by import By
import time 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import random
import requests
from IPython.display import display

In [2]:
'''url = 'https://www.dges.gov.pt/guias/indcurso.asp'
browser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options
browser.get(url)  # Open the URL in the browser
time.sleep(1)  '''

"url = 'https://www.dges.gov.pt/guias/indcurso.asp'\nbrowser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options\nbrowser.get(url)  # Open the URL in the browser\ntime.sleep(1)  "

In [3]:
'''def scrape_current_letter(browser, course_institution_data):
    """Scrape all courses & institutions from the current letter page."""
    WebDriverWait(browser, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
    )

    soup = BeautifulSoup(browser.page_source, "html.parser")
    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])
    current_course = None

    for block in all_blocks:
        classes = block.get("class", [])

        # --- Course block ---
        if "box10" in classes:
            name_tag = block.find("div", class_="lin-area-c2")
            if name_tag:
                current_course = name_tag.text.strip()
                print(f"\n📘 Course: {current_course}")

        # --- Institution block ---
        elif "lin-curso" in classes and current_course:
            link_tag = block.find("a")
            if not link_tag:
                continue

            institution = link_tag.text.strip()
            href = link_tag.get("href")

            if any(kw in institution for kw in ["Universidade", "Instituto", "Escola", "Politécnico"]):
                print(f"🏫 Institution: {institution}")

                try:
                    # Click the institution
                    inst_element = WebDriverWait(browser, 10).until(
                        EC.element_to_be_clickable((By.XPATH, f"//a[@href='{href}']"))
                    )
                    browser.execute_script("arguments[0].scrollIntoView(true);", inst_element)
                    time.sleep(0.3)
                    inst_element.click()

                    # Wait for detail page
                    WebDriverWait(browser, 15).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inside2"))
                    )
                    time.sleep(1)

                    # Parse the detail page
                    detail_soup = BeautifulSoup(browser.page_source, "html.parser")

                    # --- Extract Google Maps link ---
                    google_map = ""
                    inside_block = detail_soup.find("div", class_="inside2")
                    if inside_block:
                        map_link = inside_block.find("a", href=True, string=lambda t: t and "Mapa" in t)
                        if not map_link:
                            map_span = inside_block.find("span", class_="vislink", string=lambda t: "Mapa" in t)
                            if map_span and map_span.parent.name == "a":
                                map_link = map_span.parent
                        if map_link:
                            google_map = map_link["href"].strip()

                    print(f"🗺️ Google Maps link: {google_map if google_map else 'Not found'}")

                except Exception as e:
                    print(f"⚠️ Error scraping {institution}: {e}")
                    google_map = ""

                # Go back to the list page
                browser.back()
                time.sleep(1)
                WebDriverWait(browser, 20).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
                )

                # Store
                course_institution_data.append((current_course, institution, href, google_map))

    return course_institution_data


# --- Step 1: Scrape letter A (already selected) ---
print("\n🔤 Scraping letter: A (default)")
course_institution_data = []
course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Step 2: Click and scrape all other letters ---
letters = browser.find_elements(By.CSS_SELECTOR, "div.noprint a")
letter_links = [(a.text.strip(), a.get_attribute("href")) for a in letters if a.text.strip()]

for letter, link in letter_links:
    print(f"\n🔤 Scraping letter: {letter}")
    browser.get(link)
    time.sleep(1.5)
    course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Save results ---
df = pd.DataFrame(course_institution_data, columns=["Course", "Institution", "Link", "GoogleMaps"])'''

'def scrape_current_letter(browser, course_institution_data):\n    """Scrape all courses & institutions from the current letter page."""\n    WebDriverWait(browser, 20).until(\n        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))\n    )\n\n    soup = BeautifulSoup(browser.page_source, "html.parser")\n    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])\n    current_course = None\n\n    for block in all_blocks:\n        classes = block.get("class", [])\n\n        # --- Course block ---\n        if "box10" in classes:\n            name_tag = block.find("div", class_="lin-area-c2")\n            if name_tag:\n                current_course = name_tag.text.strip()\n                print(f"\n📘 Course: {current_course}")\n\n        # --- Institution block ---\n        elif "lin-curso" in classes and current_course:\n            link_tag = block.find("a")\n            if not link_tag:\n                continue\n\n            institution = link_tag.text.strip(

In [ ]:
df = pd.DataFrame({
    "Master Name": masters,
    "Institution": institutions,
    "About": abouts,
    "URL": program_links
})

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

# --------------------- Chrome setup ---------------------
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# --------------------- Lists ---------------------
names = []
universities = []
locations = []
durations = []
tuitions = []
abouts = []

seen = set()  # avoid duplicates

# --------------------- Loop over numbered pages ---------------------
page = 14
while True:

    url = f"https://www.mastersportal.com/search/master/portugal?page={page}"
    print(f"\n=== Loading page {page} ===")
    driver.get(url)

    # check if page has any cards
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
    except:
        print("No more results. End of pagination.")
        break

    # scroll to load all cards
    previous_len = 0
    same_count_rounds = 0
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        new_len = len(cards)

        if new_len == previous_len:
            same_count_rounds += 1
        else:
            same_count_rounds = 0

        if same_count_rounds >= 3:
            break

        previous_len = new_len

    cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
    total_cards = len(cards)
    print(f"Found {total_cards} cards.")

    if total_cards == 0:
        print("No cards on this page → stopping.")
        break

    # -------- extract each card --------
    for i in range(total_cards):

        # re-find fresh card
        cards = driver.find_elements(By.CSS_SELECTOR, "a.SearchStudyCard")
        card = cards[i]

        try:
            name = card.find_element(By.CSS_SELECTOR, "h2.StudyName").text
        except:
            name = ""

        try:
            uni = card.find_element(By.CSS_SELECTOR, "strong.OrganisationName").text
        except:
            uni = ""

        key = (name, uni)
        if key in seen or name == "":
            continue
        seen.add(key)

        print(f"\nProcessing: {name} | {uni}")

        try:
            loc = card.find_element(By.CSS_SELECTOR, "strong.OrganisationLocation").text
        except:
            loc = ""

        try:
            duration = card.find_element(By.CSS_SELECTOR, ".DurationValue").text
        except:
            duration = ""

        try:
            tuition = card.find_element(By.CSS_SELECTOR, ".TuitionValue").text
        except:
            tuition = ""

        # --- go to program page ---
        try:
            program_link = card.get_attribute("href")
            driver.get(program_link)

            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section#StudySummary")))
            time.sleep(1)

            about = driver.find_element(By.CSS_SELECTOR, "section#StudySummary p").text

            print("About extracted.")
            driver.back()
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.SearchStudyCard")))
            time.sleep(1)

        except Exception as e:
            print("Failed About:", e)
            about = ""

        names.append(name)
        universities.append(uni)
        locations.append(loc)
        durations.append(duration)
        tuitions.append(tuition)
        abouts.append(about)

    page += 1  # go to next page number

print("\n=== SCRAPING COMPLETE ===")



=== Loading page 14 ===
No more results. End of pagination.

=== SCRAPING COMPLETE ===


In [5]:
# Create DataFrame
df = pd.DataFrame({
    "Master Name": names,
    "University": universities,
    "Location": locations,
    "Duration": durations,
    "Tuition Fee": tuitions,
    "About": abouts
})
df
#df.to_csv("first_two_pages.csv", index=False, encoding="utf-8")

,Master Name,University,Location,Duration,Tuition Fee,About
0,Preschool and Primary School Education,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,This cycle of studies in Preschool and Primary...
1,Applied Mathematics and Computation,Instituto Superior Técnico,"Lisbon, Portugal",2 years,7000 EUR / year,Applied Mathematics and Computation master deg...
2,Medical Microbiology,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,2500 EUR / year,The general objective of the Master in Medical...
3,International Cybersecurity and Cyberintelligence,University of Minho,"Braga, Portugal",2 years,4000 EUR / year,The International Cybersecurity and Cyberintel...
4,Applied Biology,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,This cycle of studies is in accordance with th...
...,...,...,...,...,...,...
94,Archaeological Materials Science,The University of Évora,"Evora, Portugal",2 years,1050 EUR / year,The Archaeological Materials Science programme...
95,Communication Design,ESAD - College of Arts and Design,"Senhora da Hora, Portugal",2 years,5160 EUR / year,The Master's Degree in Communication Design fr...
96,Financial Management,Business & Economics School,"Lisbon, Portugal",2 years,,A Master’s Degree in Financial Management at B...
97,Applied Psychology,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Masters in Applied Psychology at Universit...


In [6]:
df_old = pd.read_csv("first_nine_pages.csv")
df_combined = pd.concat([df_old, df], ignore_index=True)
df_combined

,Master Name,University,Location,Duration,Tuition Fee,About
0,Sustainable Urban Mobility Transitions,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,This Sustainable Urban Mobility Transitions pr...
1,Smart Mobility Data Science and Analytics,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,This Smart Mobility Data Science and Analytics...
2,Postgraduate Program in Artificial Intelligenc...,NOVA IMS,"Lisbon, Portugal",9 months,4100 EUR / year,The Artificial Intelligence for Business Trans...
3,Postgraduate Program in Enterprise Data Scienc...,NOVA IMS,"Lisbon, Portugal",9 months,5100 EUR / year,The Enterprise Data Science and Analytics prog...
4,Economics,Nova School of Business and Economics,"Carcavelos, Portugal",1½ year,8099 EUR / year,With this Master’s in Economics from Nova Scho...
...,...,...,...,...,...,...
264,Archaeological Materials Science,The University of Évora,"Evora, Portugal",2 years,1050 EUR / year,The Archaeological Materials Science programme...
265,Communication Design,ESAD - College of Arts and Design,"Senhora da Hora, Portugal",2 years,5160 EUR / year,The Master's Degree in Communication Design fr...
266,Financial Management,Business & Economics School,"Lisbon, Portugal",2 years,,A Master’s Degree in Financial Management at B...
267,Applied Psychology,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Masters in Applied Psychology at Universit...


In [7]:
duplicates = df_combined[df_combined.duplicated(subset=["Master Name", "University"], keep=False)]

display(duplicates)

,Master Name,University,Location,Duration,Tuition Fee,About
154,Preschool and Primary School Education,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,This cycle of studies in Preschool and Primary...
155,Applied Mathematics and Computation,Instituto Superior Técnico,"Lisbon, Portugal",2 years,7000 EUR / year,Applied Mathematics and Computation master deg...
156,Medical Microbiology,Universidade Nova de Lisboa,"Lisbon, Portugal",2 years,2500 EUR / year,The general objective of the Master in Medical...
157,International Cybersecurity and Cyberintelligence,University of Minho,"Braga, Portugal",2 years,4000 EUR / year,The International Cybersecurity and Cyberintel...
158,Applied Biology,University of Madeira,"Funchal, Portugal",2 years,2000 EUR / year,This cycle of studies is in accordance with th...
159,"Humanitarian Action, Cooperation, and Developm...",University Fernando Pessoa,"Porto, Portugal",1½ year,NaN,"The Humanitarian Action, Cooperation, and Deve..."
160,Hotel Management,University of Madeira,"Funchal, Portugal",2 years,3000 EUR / year,The study cycle aimed at obtaining a Master's ...
161,"Marine Living Resources - Science, Technology ...",Universidade Nova de Lisboa,"Lisbon, Portugal",1½ year,1250 EUR / year,"The Master Marine Living Resources - Science, ..."
162,Resilience in Education,University of Lisbon,"Lisbon, Portugal",2 years,2400 EUR / year,The Resilience in Education master from Univer...
163,Management and Industrial Strategy,University of Lisbon,"Lisbon, Portugal",2 years,4850 EUR / year,The main goal of the Management and Industrial...


In [8]:
df_clean = df_combined.drop_duplicates(subset=["Master Name", "University"], keep="first")

In [9]:
df_clean

,Master Name,University,Location,Duration,Tuition Fee,About
0,Sustainable Urban Mobility Transitions,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,This Sustainable Urban Mobility Transitions pr...
1,Smart Mobility Data Science and Analytics,EIT Urban Mobility Master School,Multiple locations,2 years,4000 EUR / year,This Smart Mobility Data Science and Analytics...
2,Postgraduate Program in Artificial Intelligenc...,NOVA IMS,"Lisbon, Portugal",9 months,4100 EUR / year,The Artificial Intelligence for Business Trans...
3,Postgraduate Program in Enterprise Data Scienc...,NOVA IMS,"Lisbon, Portugal",9 months,5100 EUR / year,The Enterprise Data Science and Analytics prog...
4,Economics,Nova School of Business and Economics,"Carcavelos, Portugal",1½ year,8099 EUR / year,With this Master’s in Economics from Nova Scho...
...,...,...,...,...,...,...
264,Archaeological Materials Science,The University of Évora,"Evora, Portugal",2 years,1050 EUR / year,The Archaeological Materials Science programme...
265,Communication Design,ESAD - College of Arts and Design,"Senhora da Hora, Portugal",2 years,5160 EUR / year,The Master's Degree in Communication Design fr...
266,Financial Management,Business & Economics School,"Lisbon, Portugal",2 years,,A Master’s Degree in Financial Management at B...
267,Applied Psychology,University of Minho,"Braga, Portugal",2 years,1250 EUR / year,The Masters in Applied Psychology at Universit...


In [ ]:
df_clean.to_csv("first_thirteen_pages.csv", index=False)